In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch6. RNN기반의 Seq2Seq(스마트 번역기)</font>**
- Google Neural Nachine Translation(GNMT)
- RNN기반 Seq2Seq방식
- 인코더 입력/디코더입력(모범답안) - 디코더 출력(답안) ; 인코더와 디코더가 연결된 구조

# 1. 패키지 import 및 하이퍼 파라미터

In [4]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [8]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터 프레임을 list로 변환
print(eng_kor[:3])
print('학습할 영-한 데이터 갯수 :', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
학습할 영-한 데이터 갯수 : 110


In [14]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
k_alpha = list([ch for ch in korean])
k_alpha.sort()
print(k_alpha)

['가', '가', '각', '각', '각', '간', '감', '개', '거', '것', '게', '계', '고', '고', '관', '광', '구', '구', '굴', '규', '그', '금', '금', '금', '기', '기', '기', '기', '깊', '나', '나', '날', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '뉴', '늦', '늦', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '단', '도', '도', '도', '도', '동', '들', '람', '랑', '래', '래', '래', '래', '램', '류', '름', '름', '릎', '리', '리', '리', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '무', '무', '무', '물', '미', '바', '바', '바', '반', '방', '번', '복', '부', '부', '부', '분', '붕', '비', '뿌', '사', '사', '사', '사', '상', '색', '생', '생', '서', '선', '선', '선', '소', '소', '소', '손', '수', '쉽', '스', '스', '시', '시', '시', '식', '식', '실', '싸', '아', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '우', '우', '운', '움', '위', '유', '은', '은', '은', '은', '은', '은', '은', '을', '음', '의', '이', '이', '이', '익', '인', '인', '읽', '입', '자', '작', '장', '장', '적', '제', '좋', '주', '지', '지', '지', '지', '짜', '쪽', '쪽', '쪽', '찾', '책', '출', '칙', '크', '크', '크', '키', '탈',

In [18]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 : ',alpha)
alpha_total_size = len(alpha)
print('전체 알파벳 갯수(원핫인코딩 사이즈) :', alpha_total_size)

영어와 한글 알파벳 :  ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '가', '각', '각', '각', '간', '감', '개', '거', '것', '게', '계', '고', '고', '관', '광', '구', '구', '굴', '규', '그', '금', '금', '금', '기', '기', '기', '기', '깊', '나', '나', '날', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '뉴', '늦', '늦', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '다', '단', '도', '도', '도', '도', '동', '들', '람', '랑', '래', '래', '래', '래', '램', '류', '름', '름', '릎', '리', '리', '리', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '무', '무', '무', '물', '미', '바', '바', '바', '반', '방', '번', '복', '부', '부', '부', '분', '붕', '비', '뿌', '사', '사', '사', '사', '상', '색', '생', '생', '서', '선', '선', '선', '소', '소', '소', '손', '수', '쉽', '스', '스', '시', '시', '시', '식', '식', '실', '싸', '아', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '우', '우', '운', '움', '위', '유', '은', '은', '은', '은', '은', '은', '은', '을', '음', '의', '이', 

# 4. 문자당 num을 갖는 dict
- 전예제 : {'the':1, 'a':2,...} / {1:'the', 2:'a',...}
- {'S':0, 'E':1, ...}

In [20]:
char_to_num = {}
for i, ch in enumerate(alpha):
    # print(i, ch)
    char_to_num[ch] = i
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 30, '각': 33, '간': 34, '감': 35, '개': 36, '거': 37, '것': 38, '게': 39, '계': 40, '고': 42, '관': 43, '광': 44, '구': 46, '굴': 47, '규': 48, '그': 49, '금': 52, '기': 56, '깊': 57, '나': 59, '날': 61, '남': 62, '내': 63, '넓': 64, '녀': 65, '노': 66, '놀': 67, '농': 68, '높': 69, '뉴': 71, '늦': 73, '다': 85, '단': 86, '도': 90, '동': 91, '들': 92, '람': 93, '랑': 94, '래': 98, '램': 99, '류': 100, '름': 102, '릎': 103, '리': 107, '많': 108, '망': 109, '매': 110, '머': 111, '먼': 112, '멍': 113, '메': 114, '명': 115, '모': 116, '목': 117, '무': 121, '물': 122, '미': 123, '바': 126, '반': 127, '방': 128, '번': 129, '복': 130, '부': 133, '분': 134, '붕': 135, '비': 136, '뿌': 137, '사': 141, '상': 142, '색': 143, '생': 145, '서': 146, '선': 149, '소': 152, '손': 153, '수': 154, '쉽': 155, '스': 157, '시': 160

In [24]:
# 문자->숫자 / 숫자->문자
print('문자->숫자 : ', char_to_num.get('k'))
print('숫자->문자 : ', alpha[44])

문자->숫자 :  13
숫자->문자 :  광
